In [55]:
def get_test_data():
    import json
    examples = []
    with open("D:\\GeoTKG\\cleandata\\tie\\test.json", "r") as f:
        examples=[json.loads(line) for line in f]
    return examples

def get_prompt(text):
    sys = '''
    You are an information extraction system for geoscience and general texts. 
    Extract events and their temporal relations with arguments and time bounds.

    Rules:
    - Be literal, evidence-based; no invention. Use null if uncertain.
    - Resolve cross-sentence references (“this uplift”, “it”, etc.).
    - Use domain-appropriate terms (uplift, intrusion, orogeny, flows, glaciation, deposition, etc.).
    - Return ONLY valid JSON, no markdown or commentary.

    Temporal relations (allowed values): BEFORE, AFTER, DURING, CONTAINS, IDENTITY, EQUALS, OVERLAPS.  
    - BEFORE: event1 ends before event2 starts  
    - AFTER: event1 starts after event2 ends  
    - DURING: event1 occurs within event2  
    - CONTAINS: event1 fully contains event2  
    - IDENTITY: same event
    - OVERLAPS: partial intersection
    - EQUALS: same time span, different events

    Time normalization:
    - Use explicit ISO if present (e.g., “1998-06”, “-0120/0010”).  
    - Normalize named geological periods/epochs (e.g., “Late Cretaceous”) to their standard numeric age ranges (e.g., start_time="100.5 Ma", end_time="66 Ma") using canonical ICS chronostratigraphic boundaries. 
    - If only order is known, leave times null.  
    - If interval given (e.g., “72-66 Ma” or “2017-2018 season”), set start_time/end_time as written.

    Event arguments:
    - subject = agent/undergoer (or null)  
    - object = patient/theme (or null)  
    - event = short verb phrase (“uplift occurred”)  
    - start_time/end_time = normalized as above  
    - Keep spans concise and faithful

    Event identification:
    - Treat each distinct geologic or contemporary process/change as an event.  
    - Merge duplicates, keep most informative wording.

    JSON schema (exact):
    {
        "events": [
            {
            "id": "E1",
            "event": "string",
            "subject": "string or null",
            "object": "string or null",
            "start_time": "string or null",
            "end_time": "string or null",
            "evidence_span": "verbatim text"
            }
        ],
        "temporal_triples": [
            {
            "event1_id": "E#",
            "temp_relation": "BEFORE|AFTER|DURING|CONTAINS|IDENTITY|EQUALS|OVERLAPS",
            "event2_id": "E#",
            "evidence_span": "verbatim text"
            }
        ]
    }

    Validation:
    - Every temporal_triple must reference an event id.  
    - Use ids E1, E2… in order of appearance.  
    - evidence_span ≤30 words, quoted from passage.  
    - Output must be valid JSON (no trailing commas, no comments).
    '''
    user = f'Extract events and temporal relations from the following passage:    {text}'
    messages = [
    {"role":"system","content":sys},
    {"role":"user","content":user}
    ]
    return messages

In [ ]:
import torch, transformers

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tok = transformers.AutoTokenizer.from_pretrained(model_id)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="cuda"
)

In [ ]:
test_data = get_test_data()

In [ ]:
import json
chat_preds = []
file_num = 1
for example in test_data:
    print(f"Processing example {file_num} / {len(test_data)}")
    text = " ".join([wrd for sent in example["text"] for wrd in sent])
    input_prompt = tok.apply_chat_template(get_prompt(text), add_generation_prompt=True, tokenize=False)
    inputs = tok(input_prompt, return_tensors="pt").to(model.device)
    
    out = model.generate(**inputs, temperature=0.2, do_sample=False)

    gen_tokens = out[0, inputs.input_ids.shape[-1]+11:]
    decodings = tok.decode(gen_tokens, skip_special_tokens=True)
    prediction = decodings.strip(":\n`")
    chat_preds.append({"text":example["text"], "pred":decodings})
    file_num += 1

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Processing example 1 / 603


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


:

```
{
    "events": [
        {
            "id": "E1",
            "event": "The October Crisis occurred",
            "subject": null,
            "object": null,
            "start_time": "1970-10",
            "end_time": "1970-10",
            "evidence_span": "The October Crisis occurred in October 1970 in the province of Quebec in Canada, mainly in the Montreal metropolitan area."
        },
        {
            "id": "E2",
            "event": "Pierre Laporte and James Cross were kidnapped",
            "subject": "Members of the Front de libération du Québec (FLQ)",
            "object": "Pierre Laporte and James Cross",
            "start_time": "1970-10",
            "end_time": "1970-10",
            "evidence_span": "Members of the Front de libération du Québec (FLQ) kidnapped the provincial Deputy Premier Pierre Laporte and British diplomat James Cross."
        },
        {
            "id": "E3",
            "event": "Pierre Laporte was murdered",
            "subje

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{
    "events": [
        {
            "id": "E1",
            "event": "joked that he had a bomb on board a plane",
            "subject": "one of the passengers",
            "object": "a bomb on board the plane",
            "start_time": null,
            "end_time": null,
            "evidence_span": "one of them joked that he had a bomb on board a plane from Stockholm"
        },
        {
            "id": "E2",
            "event": "overheard that one passenger told his friends he had a bomb on board the plane",
            "subject": "the crew",
            "object": "one passenger telling his friends",
            "start_time": null,
            "end_time": null,
            "evidence_span": "the crew has overheard that one passenger told his friends he had a bomb on board the plane"
        },
        {
            "id": "E3",
            "event": "carry on to Malaga and landed without incident",
            "subject": "the captain of the flight",
            "object": null

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


:

```
{
    "events": [
        {
            "id": "E1",
            "event": "Tomorrow, the board of supervisors of Loudon county, Virginia, will vote on whether a school now located in Mount Vernon can relocate to their county.",
            "subject": "board of supervisors of Loudon county, Virginia",
            "object": "school",
            "start_time": null,
            "end_time": null,
            "evidence_span": "Tomorrow the board of supervisors of Loudon county, Virginia, will vote on whether a school now located in Mount Vernon can relocate to their county."
        },
        {
            "id": "E2",
            "event": "The Islamic Saudi Academy, funded by the government of Saudi Arabia, can move to Ashburn village, Virginia.",
            "subject": "Islamic Saudi Academy",
            "object": "Ashburn village, Virginia",
            "start_time": null,
            "end_time": null,
            "evidence_span": "The board will decide whether the Islamic Saudi A

KeyboardInterrupt: 

In [ ]:
with open("llama3-8B-preds.json", 'w') as json_file:
    for sample in chat_preds:
        json_file.write(json.dumps(sample)+"\n")

In [61]:
import isodate

def gentext_to_iso8601(gentext: str):
    parsers = {
        isodate.parse_date,
        isodate.parse_datetime,
        isodate.parse_time,
        isodate.parse_duration,
    }
    for parser in parsers:
        try:
            output = parser(gentext)
            if output is not None:
                return output
        except Exception:
            continue
        return None
    
def get_start_end_times(event_times: list):
    for time in event_times:
        isoobj = gentext_to_iso8601(time)
        print(isoobj)

In [69]:
events = {}
times = {}
event_quins = {}
ee_trips = []
ets = {}

for instance in test_data[0]['instances']:
    instance_id = instance["id"]
    if instance["type"] == "EVENT":
        events[instance_id] = instance
    else:
        times[instance_id] = instance

for ee in test_data[0]["ee_temprels"]:
    ee_trips.append({"event1_id":ee["e1"], "temp_relation":ee["rel"], "event2_id":ee["e2"]})

for et in test_data[0]["event_times"]:
    evid = et["event"] 
    value = times[et["time"]]['value']
    if evid not in ets:
        ets[evid] = [value]
    else:
        ets[evid].append(value)

#for eid, event in events.items():




KeyError: 'value'

In [67]:
for evti in ets:
    get_start_end_times(ets[evti])

None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
